# Connect to Neo4j — deluxe-analyze

Replace `NEO4J_IP` with the static IP from `terraform output neo4j_static_ip`.
Add your Colab/notebook IP to `allowed_neo4j_ips` in `infra/terraform.tfvars` and re-apply.

In [ ]:
import os
from neo4j import GraphDatabase

NEO4J_IP = os.environ.get("NEO4J_IP", "REPLACE_ME")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "REPLACE_ME")

driver = GraphDatabase.driver(
    f"bolt://{NEO4J_IP}:7687",
    auth=("neo4j", NEO4J_PASSWORD),
)
driver.verify_connectivity()
print("Connected to Neo4j")

In [ ]:
with driver.session(database="neo4j") as session:
    # Basic connectivity check
    result = session.run("MATCH (n) RETURN count(n) AS total")
    print(f"Total nodes: {result.single()['total']}")

    # Degree centrality via GDS
    result = session.run("""
        CALL gds.degree.stream('social-graph')
        YIELD nodeId, score
        RETURN gds.util.asNode(nodeId).id AS user_id, score AS degree
        ORDER BY degree DESC
        LIMIT 10
    """)
    for row in result:
        print(f"  {row['user_id']}: degree={row['degree']}")